In [1]:
import pandas as pd

train_df = pd.read_csv("../data/processed/train_processed.csv")
val_df = pd.read_csv("../data/processed/val_processed.csv")

print(train_df.shape, val_df.shape)
train_df.head()

(143613, 2) (15958, 2)


,comment_text,harassment
0,", 25 March 2013 (UTC)\n\nThat's some strange i...",0
1,"""\n{| style=""""background-color: #F5FFFA; paddi...",0
2,You Republic of Turkey and supporters therof a...,1
3,I have commented on them in my unblock reason....,0
4,Very interesting comments about the purpose an...,0


In [3]:
train_df = train_df.rename(columns={"comment_text": "text"})
val_df = val_df.rename(columns={"comment_text": "text"})

print(train_df.head())

                                                text  harassment
0  , 25 March 2013 (UTC)\n\nThat's some strange i...           0
1  "\n{| style=""background-color: #F5FFFA; paddi...           0
2  You Republic of Turkey and supporters therof a...           1
3  I have commented on them in my unblock reason....           0
4  Very interesting comments about the purpose an...           0


In [4]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['text', 'harassment'],
    num_rows: 143613
})
Dataset({
    features: ['text', 'harassment'],
    num_rows: 15958
})


In [5]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer loaded successfully")

Tokenizer loaded successfully


In [6]:
def tokenize_function(example):

    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)


print("Tokenization complete")

Map: 100%|█████████████████████████████████████████████████████| 15958/15958 [00:01<00:00, 10778.55 examples/s]

Tokenization complete


In [7]:
train_dataset = train_dataset.rename_column("harassment", "labels")
val_dataset = val_dataset.rename_column("harassment", "labels")

print(train_dataset)

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 143613
})


In [8]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("Model loaded")

Loading weights: 100%|████████| 199/199 [00:00<00:00, 591.93it/s, Materializing param=bert.pooler.dense.weight]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized becau

Model loaded


In [9]:
import torch

device = torch.device("cuda")

model.to(device)

print("Model running on:", device)
print("GPU:", torch.cuda.get_device_name(0))

Model running on: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [10]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    print("GPU NOT detected")

CUDA available: True
Current device: 0
Device name: NVIDIA GeForce RTX 4060 Laptop GPU


In [12]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="../models/bert-harassment",

    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    num_train_epochs=1,

    eval_strategy="epoch",

    save_strategy="epoch",

    logging_dir="../logs",

    load_best_model_at_end=True,

    report_to=None

)

print("Training config ready")

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Training config ready


In [14]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

print("Metrics function ready")

Metrics function ready


In [16]:
train_dataset

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 143613
})

In [1]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="../models/bert-harassment",

    learning_rate=2e-5,

    per_device_train_batch_size=8,   # ✅ reduced for stability
    per_device_eval_batch_size=8,

    num_train_epochs=1,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none",

    fp16=True,   # ✅ MASSIVE improvement (faster + safer)

    logging_steps=200

)

print("TrainingArguments ready")

D:\hackathons & other projects\ai-chat-harassment-detection\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


TrainingArguments ready


In [2]:
pip install ipywidgets

   ---------------------------------------- 0.0/914.9 kB ? eta -:--:--
   ---------------------------------------- 914.9/914.9 kB 6.9 MB/s  0:00:00
   ---------------------------------------- 0.0/2.2 MB ? eta -:--:--
   ---------------------------------------- 2.2/2.2 MB 17.6 MB/s  0:00:00

   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]

Note: you may need to restar

In [4]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer loaded")

Tokenizer loaded


In [5]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("Model loaded")

Loading weights: 100%|█| 199/199 [00:00<00:00, 516.67it/s, Materializing param=bert.pooler.dens
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from 

Model loaded


In [6]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print("Model moved to:", device)

Model moved to: cuda


In [10]:
import os

base = r"D:\hackathons & other projects\ai-chat-harassment-detection\data"

print("RAW folder:")
print(os.listdir(os.path.join(base, "raw")))

print("\nPROCESSED folder:")
print(os.listdir(os.path.join(base, "processed")))

RAW folder:
['sample_submission.csv', 'test.csv', 'test_labels.csv', 'train.csv']

PROCESSED folder:
['train_processed.csv', 'val_processed.csv']


In [11]:
import pandas as pd

train_df = pd.read_csv(
    r"D:\hackathons & other projects\ai-chat-harassment-detection\data\processed\train_processed.csv"
)

val_df = pd.read_csv(
    r"D:\hackathons & other projects\ai-chat-harassment-detection\data\processed\val_processed.csv"
)

print("Train shape:", train_df.shape)
print("Val shape:", val_df.shape)

train_df.head()

Train shape: (143613, 2)
Val shape: (15958, 2)


,comment_text,harassment
0,", 25 March 2013 (UTC)\n\nThat's some strange i...",0
1,"""\n{| style=""""background-color: #F5FFFA; paddi...",0
2,You Republic of Turkey and supporters therof a...,1
3,I have commented on them in my unblock reason....,0
4,Very interesting comments about the purpose an...,0


In [12]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset   = Dataset.from_pandas(val_df)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['comment_text', 'harassment'],
    num_rows: 143613
})
Dataset({
    features: ['comment_text', 'harassment'],
    num_rows: 15958
})


In [13]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print("Tokenizer loaded")

Tokenizer loaded


In [17]:
print(train_dataset.column_names)

['comment_text', 'harassment']


In [18]:
def tokenize_function(example):
    return tokenizer(
        example["comment_text"],   # ✅ fixed column name
        truncation=True,
        padding="max_length",
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset   = val_dataset.map(tokenize_function, batched=True)

print("Tokenization complete")

Map: 100%|██████████████████████████████████████| 15958/15958 [00:01<00:00, 8758.63 examples/s]

Tokenization complete


In [19]:
train_dataset = train_dataset.rename_column("harassment", "labels")
val_dataset   = val_dataset.rename_column("harassment", "labels")

print(train_dataset.column_names)

['comment_text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask']


In [20]:
train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "labels"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "token_type_ids", "labels"]
)

print("PyTorch format set")

PyTorch format set


In [21]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

print("Model loaded")

Loading weights: 100%|█| 199/199 [00:00<00:00, 498.59it/s, Materializing param=bert.pooler.dens
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from 

Model loaded


In [22]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)

print("Using device:", device)

Using device: cuda


In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="../models/bert-harassment",

    learning_rate=2e-5,

    per_device_train_batch_size=8,   # safe for your RTX 4060 laptop
    per_device_eval_batch_size=8,

    num_train_epochs=1,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    report_to="none",

    fp16=True,   # faster + less VRAM

    logging_steps=200

)

print("TrainingArguments ready")

TrainingArguments ready


In [24]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):

    logits, labels = eval_pred

    predictions = logits.argmax(axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="binary"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

print("Metrics ready")

Metrics ready


In [ ]:
from transformers import Trainer

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    compute_metrics=compute_metrics

)

trainer.train()

Epoch,Training Loss,Validation Loss
